In [ ]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model
from datasets import Dataset
import os
os.environ["WANDB_DISABLED"] = "true"

# Check if CUDA is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load the dataset
with open("data/attribute.json", "r") as f:
    attribute_data = json.load(f)
with open("data/cfe.json", "r") as f:
    cfe_data = json.load(f)
data = attribute_data + cfe_data

# Prepare the dataset for fine-tuning
def format_example(example):
    prompt = f"""<|begin_of_text|>User: {example['user']} <|end_of_text|>
Assistant: {example['parsed']} <|end_of_text|>"""
    return {"text": prompt}

# Convert to Hugging Face Dataset
formatted_data = [format_example(item) for item in data]
dataset = Dataset.from_list(formatted_data)

# Load tokenizer and model
model_name = "meta-llama/Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# Set padding token
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

# Tokenize the dataset and prepare labels
def tokenize_function(examples):
    tokenized = tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)
    # Create labels by copying input_ids
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# Split dataset into train and eval
train_test_split = tokenized_dataset.train_test_split(test_size=0.1)
train_dataset = train_test_split["train"]
eval_dataset = train_test_split["test"]

# Configure LoRA for efficient fine-tuning
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

# Define training arguments
training_args = TrainingArguments(
    output_dir="./llama_finetuned",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_steps=10,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    evaluation_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    load_best_model_at_end=True,
    fp16=True,
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

# Fine-tune the model
trainer.train()

# Save the fine-tuned model
model.save_pretrained("./llama_finetuned/final")
tokenizer.save_pretrained("./llama_finetuned/final")

Using device: cuda


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# Check if CUDA is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load the fine-tuned model and tokenizer
model_path = "./llama_finetuned/final"
base_model_name = "meta-llama/Llama-3.1-8B-Instruct"
# base_model_name = "meta-llama/Llama-3.2-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_path)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
model = PeftModel.from_pretrained(base_model, model_path)
model.eval()

# Function to generate text
def generate_response(user_input):
    # Format the input as it was during training
    prompt = f"""<|begin_of_text|>User: {user_input} <|end_of_text|>
Assistant: """
    
    # Tokenize the input
    inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True, max_length=128)
    inputs = {key: val.to(device) for key, val in inputs.items()}
    
    # Generate response
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=20,  # Limit the length of generated text
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            do_sample=False,  # Greedy decoding for deterministic output
        )
    
    # Decode the generated text
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=False)
    
    # Extract the assistant's response (after "Assistant: ")
    response = generated_text.split("Assistant: ")[-1].strip()
    return response

# Example input
user_input = "What are the most important features for this prediction?"
response = generate_response(user_input)
print(f"User: {user_input}")
print(f"Model Response: {response}")

# Additional example inputs
example_inputs = [
    "Why do you predict this sample?",
    "Explain the prediction using SHAP",
    "What features matter most?"
]
example_inputs = [
        "What are the most important features for this prediction?",
        "Why do you predict this sample?",
        "Explain the prediction using SHAP",
        "What features matter most?",
        "Why do you predict it?",
        "What does instance have to do in order to change the prediction?"
        
    ]
for input_text in example_inputs:
    response = generate_response(input_text)
    print(f"\nUser: {input_text}")
    print(f"Model Response: {response}")

Using device: cuda


Loading checkpoint shards: 100%|██████████| 4/4 [00:14<00:00,  3.65s/it]



User: What are the most important features for this prediction?
Model Response: [SHAP(input,label)] <|end_of_text|><|eot_id|>

User: What are the most important features for this prediction?
Model Response: [SHAP(input,label)] <|end_of_text|><|eot_id|>

User: What are the most important features for this prediction?
Model Response: [SHAP(input,label)] <|end_of_text|><|eot_id|>

User: Why do you predict this sample?
Model Response: [SHAP(input,label)] <|end_of_text|><|eot_id|>

User: Why do you predict this sample?
Model Response: [SHAP(input,label)] <|end_of_text|><|eot_id|>

User: Explain the prediction using SHAP
Model Response: [SHAP(input,label)] <|end_of_text|><|eot_id|>

User: Explain the prediction using SHAP
Model Response: [SHAP(input,label)] <|end_of_text|><|eot_id|>

User: What features matter most?
Model Response: [SHAP(input,label)] <|end_of_text|><|eot_id|>

User: What features matter most?
Model Response: [SHAP(input,label)] <|end_of_text|><|eot_id|>

User: Why do you pr

In [ ]:
import os
import json
import yaml
from langchain.document_loaders import TextLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
import torch
from huggingface_hub import login
from peft import PeftModel

# Authenticate with HuggingFace (required for gated LLaMA models)
# login(token=os.getenv("HF_TOKEN"))  # Or replace with your token directly

# Step 1: Load metadata from JSON/YAML files
def load_metadata_files(directory: str):
    documents = []
    for filename in os.listdir(directory):
        if filename.endswith('.json') or filename.endswith('.yaml') or filename.endswith('.yml'):
            filepath = os.path.join(directory, filename)
            with open(filepath, 'r') as f:
                if filename.endswith('.json'):
                    data = json.load(f)
                else:
                    data = yaml.safe_load(f)
                text = json.dumps(data, indent=2)  # Convert to string for consistency
                documents.append(text)
    return documents

# Step 2: Prepare documents for RAG (split into chunks if large)
def prepare_documents(raw_texts):
    splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    docs = []
    for text in raw_texts:
        split_docs = splitter.create_documents([text])
        docs.extend(split_docs)
    return docs

# Step 3: Build vector store for retrieval
def build_vector_store(docs):
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vector_store = FAISS.from_documents(docs, embeddings)
    return vector_store

# Step 4: Set up LLaMA 3.2 as the LLM
def setup_llm():
    base_model_name = "meta-llama/Llama-3.1-8B-Instruct"
    model_path = "llama_finetuned/final"
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto"  # Automatically maps to GPU/CPU
    )
    model = PeftModel.from_pretrained(base_model, model_path)
    llm_pipeline = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=200,
        device_map="auto",
        return_full_text=False
    )
    llm = HuggingFacePipeline(pipeline=llm_pipeline)
    return llm

# Step 5: Set up RAG chain with custom prompt
def setup_rag_chain(vector_store, llm):
    prompt_template = """<|begin_of_text|>User: {context} {question} <|end_of_text|>Assistant: """
    PROMPT = PromptTemplate(template=prompt_template, input_variables=["context", "question"])
    
    chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=vector_store.as_retriever(search_kwargs={"k": 3}),  # Retrieve top 3 chunks
        chain_type_kwargs={"prompt": PROMPT}
    )
    return chain

# Function to ask a question
def ask_question(query):
    metadata_dir = "./metadata/dataset"  # Change to your directory
    raw_texts = load_metadata_files(metadata_dir)
    if not raw_texts:
        return "No metadata files found."
    
    docs = prepare_documents(raw_texts)
    vector_store = build_vector_store(docs)
    llm = setup_llm()
    rag_chain = setup_rag_chain(vector_store, llm)
    
    response = rag_chain.run(query)
    return response

# Example usage
query = "What is the dataset about?"
answer = ask_question(query)
print("Question:", query)
print("Answer:", answer)

/home/nguyenv9/miniconda3/envs/xagent-llms-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:06<00:00,  3.40s/it]
Device set to use cuda:0


LLaMA 3.2 Agent ready. Ask questions about the metadata (type 'exit' to quit).
